In [ ]:
pip install transformers accelerate torch sentence-transformers rank_bm25 faiss-cpu langchain-text-splitters tiktoken

In [ ]:
import re
import numpy as np
import faiss
import torch
import logging
from typing import List, Dict
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from langchain_text_splitters import RecursiveCharacterTextSplitter

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# ==========================================
# 1. Локальный провайдер LLM (Вместо vLLM API)
# ==========================================
class LocalLLM:
    def __init__(self, model_id="Qwen/Qwen2.5-3B-Instruct"):
        logger.info(f"Загрузка локальной LLM: {model_id}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        # Загружаем с 16-битной точностью для экономии видеопамяти
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id, 
            device_map="auto", 
            torch_dtype=torch.float16
        )
        logger.info("LLM успешно загружена!")

    def get_response(self, prompt: str, system_prompt: str = "You are a helpful assistant.") -> str:
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ]
        text = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)
        
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs, 
                max_new_tokens=512,
                temperature=0.1, # Низкая температура для RAG (меньше галлюцинаций)
                do_sample=True
            )
        
        # Отрезаем промпт от сгенерированного ответа
        response = self.tokenizer.decode(outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True)
        return response.strip()

# ==========================================
# 2. Модернизированный RAG Pipeline
# ==========================================
class AdvancedRAGPipeline:
    def __init__(self, llm: LocalLLM, embedding_model="paraphrase-multilingual-MiniLM-L12-v2"):
        self.llm = llm # Инжектим нашу локальную LLM
        
        logger.info(f"Загрузка эмбеддинг модели: {embedding_model}")
        self.embedder = SentenceTransformer(embedding_model)
        self.embed_dim = self.embedder.get_sentence_embedding_dimension()
        
        logger.info("Загрузка Cross-Encoder...")
        self.cross_encoder = CrossEncoder('cross-encoder/mmarco-mMiniLMv2-L12-H384-v1')

        self.indices = {
            "chunk": self._create_hnsw_index(),
            "cleaned": self._create_hnsw_index(),
            "q2c": self._create_hnsw_index(),
            "summary": self._create_hnsw_index()
        }
        
        self.bm25_corpus = []
        self.bm25 = None
        self.chunks_meta = [] 
        
        self.weights = {"bm25": 0.30, "chunk": 0.25, "cleaned": 0.20, "q2c": 0.15, "summary": 0.10}
    
    def _create_hnsw_index(self):
        index = faiss.IndexHNSWFlat(self.embed_dim, 32, faiss.METRIC_INNER_PRODUCT) 
        index.hnsw.efConstruction = 200
        index.hnsw.efSearch = 64
        return index

    def _clean_text(self, text: str) -> str:
        text = text.lower()
        text = re.sub(r'[^\w\s\d]', ' ', text)
        return re.sub(r'\s+', ' ', text).strip()

    def process_documents(self, documents: List[str]):
        splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_size=512, chunk_overlap=256)
        
        raw_chunks = []
        for doc_id, doc_text in enumerate(documents):
            splits = splitter.split_text(doc_text)
            for i, split in enumerate(splits):
                raw_chunks.append({
                    "doc_id": doc_id,
                    "chunk_id": len(self.chunks_meta) + len(raw_chunks),
                    "text": split,
                    "prev_text": splits[i-1] if i > 0 else "",
                    "next_text": splits[i+1] if i < len(splits)-1 else ""
                })

        processed_chunks = []
        for c in raw_chunks:
            c['cleaned_text'] = self._clean_text(c['text'])
            
            q2c_prompt = f"Прочитай фрагмент и сгенерируй 3 поисковых вопроса к нему:\n\n{c['text']}"
            c['q2c'] = self.llm.get_response(q2c_prompt, "Ты ассистент, генерирующий вопросы. Пиши только вопросы.")
            
            context_text = f"ПРЕДЫДУЩАЯ: {c['prev_text']}\nТЕКУЩАЯ: {c['text']}\nСЛЕДУЮЩАЯ: {c['next_text']}"
            summary_prompt = f"Суммируй основную мысль в 1-2 предложения с фокусом на фактах:\n{context_text}"
            c['context_summary'] = self.llm.get_response(summary_prompt, "Ты суммаризатор.")
            
            processed_chunks.append(c)

        vec_chunk = self.embedder.encode([c['text'] for c in processed_chunks], normalize_embeddings=True)
        vec_cleaned = self.embedder.encode([c['cleaned_text'] for c in processed_chunks], normalize_embeddings=True)
        vec_q2c = self.embedder.encode([c['q2c'] for c in processed_chunks], normalize_embeddings=True)
        vec_summary = self.embedder.encode([c['context_summary'] for c in processed_chunks], normalize_embeddings=True)

        self.indices["chunk"].add(np.array(vec_chunk, dtype=np.float32))
        self.indices["cleaned"].add(np.array(vec_cleaned, dtype=np.float32))
        self.indices["q2c"].add(np.array(vec_q2c, dtype=np.float32))
        self.indices["summary"].add(np.array(vec_summary, dtype=np.float32))

        self.bm25_corpus.extend([c['cleaned_text'].split() for c in processed_chunks])
        self.bm25 = BM25Okapi(self.bm25_corpus)
        self.chunks_meta.extend(processed_chunks)

    def _min_max_normalize(self, scores: np.ndarray) -> np.ndarray:
        if len(scores) == 0: return scores
        min_val, max_val = np.min(scores), np.max(scores)
        if max_val - min_val == 0: return np.ones_like(scores)
        return (scores - min_val) / (max_val - min_val)

    def search(self, query: str, top_k: int = 50, final_top_k: int = 5):
        query_cleaned = self._clean_text(query)
        q_vec = self.embedder.encode([query], normalize_embeddings=True).astype(np.float32)
        q_vec_cleaned = self.embedder.encode([query_cleaned], normalize_embeddings=True).astype(np.float32)

        results_dict = {i: {"scores": {k: 0.0 for k in self.weights.keys()}} for i in range(len(self.chunks_meta))}

        D_chunk, I_chunk = self.indices["chunk"].search(q_vec, top_k)
        D_clean, I_clean = self.indices["cleaned"].search(q_vec_cleaned, top_k)
        D_q2c, I_q2c = self.indices["q2c"].search(q_vec, top_k)
        D_sum, I_sum = self.indices["summary"].search(q_vec, top_k)

        bm25_scores = self.bm25.get_scores(query_cleaned.split())
        bm25_top_indices = np.argsort(bm25_scores)[::-1][:top_k]
        
        def aggregate_scores(indices, distances, key):
            valid_idx = indices[0] != -1
            idx_array, dist_array = indices[0][valid_idx], distances[0][valid_idx]
            norm_dists = self._min_max_normalize(dist_array)
            for i, chunk_idx in enumerate(idx_array):
                results_dict[chunk_idx]["scores"][key] = norm_dists[i]

        aggregate_scores(I_chunk, D_chunk, "chunk")
        aggregate_scores(I_clean, D_clean, "cleaned")
        aggregate_scores(I_q2c, D_q2c, "q2c")
        aggregate_scores(I_sum, D_sum, "summary")

        norm_bm25 = self._min_max_normalize(bm25_scores[bm25_top_indices])
        for i, chunk_idx in enumerate(bm25_top_indices):
            results_dict[chunk_idx]["scores"]["bm25"] = norm_bm25[i]

        fusion_results = []
        for chunk_idx, data in results_dict.items():
            final_score = sum(data["scores"][k] * self.weights[k] for k in self.weights.keys())
            if final_score > 0: fusion_results.append((final_score, chunk_idx))

        top_m = sorted(fusion_results, key=lambda x: x[0], reverse=True)[:top_k]

        # Cross-Encoder
        cross_inp = [[query, self.chunks_meta[idx]['text']] for _, idx in top_m]
        cross_scores = self.cross_encoder.predict(cross_inp)
        
        final_results = sorted(zip(cross_scores, [idx for _, idx in top_m]), key=lambda x: x[0], reverse=True)
        return [self.chunks_meta[idx] for _, idx in final_results[:final_top_k]]

    def answer_query(self, query: str):
        retrieved_chunks = self.search(query, top_k=20, final_top_k=3)
        context_str = "\n\n".join([f"[Фрагмент {i+1}]: {chunk['text']}" for i, chunk in enumerate(retrieved_chunks)])
        
        prompt = f"""Опирайся ТОЛЬКО на предоставленные фрагменты. Если ответа нет, скажи "Я не знаю".
Контекст:
{context_str}

Вопрос: {query}"""

        response = self.llm.get_response(prompt, "Ты точный RAG-ассистент.")
        return response, retrieved_chunks

# ==========================================
# 3. Модуль Офлайн Метрик (Оценка)
# ==========================================
class RAGEvaluator:
    def __init__(self, rag_pipeline: AdvancedRAGPipeline):
        self.rag = rag_pipeline

    def evaluate(self, eval_dataset: List[Dict]):
        print(f"\n[ОЦЕНКА] Начало оценки на {len(eval_dataset)} примерах. Пожалуйста, подождите (LLM думает)...")
        
        metrics = {"hit_rate": 0.0, "mrr": 0.0, "faithfulness": 0.0}
        
        for i, item in enumerate(eval_dataset):
            print(f"  -> Обработка примера {i+1}/{len(eval_dataset)}: '{item['query']}'")
            query = item["query"]
            target_idx = item["relevant_doc_idx"]
            
            # 1. Замеряем поиск
            retrieved = self.rag.search(query, top_k=20, final_top_k=5)
            retrieved_ids = [chunk['doc_id'] for chunk in retrieved]
            
            # Hit Rate
            if target_idx in retrieved_ids:
                metrics["hit_rate"] += 1
            
            # MRR
            try:
                rank = retrieved_ids.index(target_idx) + 1
                metrics["mrr"] += 1.0 / rank
            except ValueError:
                pass

            # 2. Замеряем генерацию (LLM-as-a-Judge)
            answer, _ = self.rag.answer_query(query)
            context_used = "\n".join([c['text'] for c in retrieved])
            
            judge_prompt = f"""Оцени, насколько сгенерированный ответ опирается на предоставленный контекст.
Контекст: {context_used}
Ответ: {answer}

Выведи ТОЛЬКО цифру 1 (если ответ строго следует контексту) или 0 (если в ответе есть галлюцинации или инфа не из контекста)."""
            
            judge_score_str = self.rag.llm.get_response(judge_prompt, "Ты строгий судья. Отвечай только цифрой 0 или 1.")
            
            if "1" in judge_score_str:
                metrics["faithfulness"] += 1
                print(f"     [+] Faithfulness: ОК (Опирается на контекст)")
            else:
                print(f"     [-] Faithfulness: ПРОВАЛ (Возможна галлюцинация)")

        total = len(eval_dataset)
        metrics = {k: round((v / total) * 100, 2) for k, v in metrics.items()}
       
        metrics["mrr"] = round(metrics["mrr"] / 100, 4) 
        
        print("\n=== ИТОГОВЫЕ ОФЛАЙН МЕТРИКИ ===")
        print(f"Hit Rate@5:   {metrics['hit_rate']}% (Документ найден в топе)")
        print(f"MRR:          {metrics['mrr']} (Средний ранг нужного документа)")
        print(f"Faithfulness: {metrics['faithfulness']}% (Ответы без галлюцинаций)")
        
        return metrics

# ==========================================
# 4. Точка входа
# ==========================================
if __name__ == "__main__":
    
    local_llm = LocalLLM(model_id="Qwen/Qwen2.5-3B-Instruct") 
    
    rag = AdvancedRAGPipeline(llm=local_llm)
    
    sample_documents = [
        """Искусственный интеллект (ИИ) в медицине используется для диагностики заболеваний, разработки новых лекарств и персонализированного лечения. Алгоритмы машинного обучения могут анализировать рентгеновские снимки с точностью, превышающей человеческую. Например, в 2023 году система от Google Health показала снижение ложноположительных диагнозов рака груди на 5.7%.""",
        """Квантовые компьютеры используют кубиты, которые могут находиться в состоянии суперпозиции. Это позволяет им решать определенные классы задач, такие как факторизация больших чисел или моделирование сложных молекул, экспоненциально быстрее классических компьютеров. Алгоритм Шора — один из самых известных квантовых алгоритмов."""
    ]
    
    print("\n--- СТАРТ ИНДЕКСАЦИИ ---")
    rag.process_documents(sample_documents)
    
    print("\n--- ТЕСТОВЫЙ ЗАПРОС ---")
    user_query = "На сколько процентов система Google Health снизила ошибки в диагностике?"
    answer, sources = rag.answer_query(user_query)
    print(f"Ответ:\n{answer}")
    
    print("\n--- РАСЧЕТ ОФЛАЙН МЕТРИК ---")
    evaluator = RAGEvaluator(rag)
    
    eval_dataset = [
        {
            "query": "Что показала система Google Health в 2023 году?",
            "ground_truth_answer": "Система снизила ложноположительные диагнозы рака груди на 5.7%.",
            "relevant_doc_idx": 0 
        },
        {
            "query": "Для чего квантовым компьютерам нужны кубиты?",
            "ground_truth_answer": "Для нахождения в состоянии суперпозиции и решения задач экспоненциально быстрее классических ПК.",
            "relevant_doc_idx": 1 
        }
    ]
    

    final_metrics = evaluator.evaluate(eval_dataset)